# **[실습]**

### 실습 목표

Contextual Retrieval을 실제 문서에 적용하여 검색 성능을 개선합니다.

### 난이도별 가이드

**기본 난이도:**
- 제공된 샘플 문서에 Contextual Retrieval 적용
- 일반 검색 vs Contextual 검색 성능 비교
- 3개 이상의 쿼리로 테스트

**중급 난이도:**
- 자체 문서(예: 기술 문서, 위키피디아) 활용
- Hybrid 검색 (Embedding + BM25) 구현
- 가중치 조합 실험 (0.3:0.7, 0.5:0.5, 0.7:0.3)

**고급 난이도:**
- 다국어 문서에 Contextual Retrieval 적용
- Reranker와 결합하여 최적 파이프라인 구성
- 검색 성능 지표 (HitRate, MRR) 측정 및 비교

`(1) 문서 준비 및 청킹`

자신의 문서를 로드하고 청크로 분할합니다.

0. 환경 설정 및 준비

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from pprint import pprint

1. 문서 준비/로드

```text```

In [3]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    '/Users/greenpianorabbit/edu/modu_llm5/docs/key_data/KTS_access_infra_improve_guide_vision_16h32_dpi300.txt',
    encoding='utf-8'
)
docs = loader.load()

# metadata source를 의미있는 이름으로 덮어쓰기
docs[0].metadata['source'] = '국채전문유통시장(KTS) 접속인프라 및 관리체계 개선 안내자료 (2023.6)'

In [4]:
docs[0].page_content

'```\n국채전문유통시장(KTS)\n접속인프라 및 관리체계 개선 안내자료\n\n2023. 6\n\n한국거래소\n채권시장부\n\n회원 및 시스템 개발사 등의 인프라 개발에 필요한 시장 구조 등 전반적인 일반사항과 제도개선 사항을 포함하고 있습니다.\n세부 제도 및 개발 방향은 변경될 수 있습니다.\n```\n\n---\n\n### 2 주요 추진 일정\n\n- 회원사의 시스템 개발 부담 및 수용성을 고려해 단계적 개발 및 전환 추진\n- 각 회원사(증권, 은행)는 \'23.7월 말까지 하던 시스템 개발 및 전환일정 중 선택하여 KRX로 통보\n\n#### 회원사 시스템 개발 일정(안)\n\n| 구분          | I/F 일반매매 | I/F 시장조정 | 안전장치 |\n| -------------- | ------------- | ------------- | --------- |\n| 1. 사전개발 참여(~\'23년말) | ●           | -           | -         |\n| 2. 본개발 참여(~\'24년 1차 정기편) | ●           | ●           | ●         |\n| 3. 후발 개발(본단계 완료 후) | -           | ●           | ●         |\n\n1. (사전개발) I/F 시장조정 및 안전장치 중심의 본개발: 가동 이전 I/F를 통한 국제시장의 일반 매매기능 등 구축 및 정비 가능\n   - (사전개발 변별) I/F 일반매매와 기반 구축, 신탁 및 외국인 매매기반 정비\n   - 사전개발에 참여하는 회원사도 본개발 참여 필요\n\n2. (본개발) I/F 일반 매매기능 등 + 시장조정 및 안전장치 개발\n   - 본개발 사항의 가동시기는 \'24년 상반기(1차 정기개편) 예정\n\n3. (후발 개발) \'24년 상반기 본개발 이후에 개발하는 유형\n   - 구축조건 개발기한은 현재 미정이나 UI기능을 전지적으로 축소 예정(전체 회원사 개발일정 등 고려하여 추후 사전 공지)\n

## 2. 텍스트 분할/문서 청킹 (Text Splitting)

In [5]:
# 전체 길이 확인
print(f"전체 텍스트: {len(docs[0].page_content):,}자")

# 문단 길이 분포 파악 (--- 구분자로 나뉜 섹션들)
sections = docs[0].page_content.split('---')
lengths = [len(s.strip()) for s in sections if len(s.strip()) > 0]
print(f"섹션 수: {len(lengths)}")
print(f"섹션 평균 길이: {sum(lengths)//len(lengths)}자")
print(f"섹션 최소/최대: {min(lengths)} / {max(lengths)}자")

# 섹션 평균 211자는 너무 작아 — 단독 청크로 쓰면 맥락이 부족해. 2~3개 섹션을 묶어서 하나의 청크로 만드는 게 적합
# ---를 separator에 넣지 않는 이유: 넣으면 평균 211자짜리 276개 조각이 그대로 나와서 오히려 너무 작아짐
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,        # 섹션 평균 211자 × 2~3개 묶음
    chunk_overlap=100,     # 섹션 경계에서 맥락 유지
    separators=["\n\n", "\n", ""],  # --- 는 제외: 너무 잘게 쪼개짐 방지
)

chunks = text_splitter.split_documents(docs)
print(f"청크 수: {len(chunks)}")  # 예상 약 120~150개

전체 텍스트: 60,155자
섹션 수: 271
섹션 평균 길이: 208자
섹션 최소/최대: 1 / 2846자
청크 수: 140


- OpenAI text-embedding-3-small 임베딩 모델 사용 (Intel Mac CPU 속도 문제로 bge-m3 대신 선택)
- 같은 모델에 맞는 토크나이저 사용 권장! ⭐⭐⭐⭐⭐ (Claude ai)
  1. 정확성 중요 - 법률 용어 정확히 처리
  2. 긴 문서 - max_length 초과 방지
  3. 프로덕션급 - 나중에 리팩토링 불필요
- text-embedding-3-small → tiktoken (cl100k_base) 사용

In [6]:
# Step 1: 임베딩 모델 결정 (OpenAI - Intel Mac CPU 속도 문제 회피)
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 2: 같은 모델의 토크나이저 가져오기 (tiktoken)
import tiktoken
tokenizer = tiktoken.encoding_for_model("text-embedding-3-small")  # cl100k_base

# 토크나이저 설정 확인
print(f"토크나이저: {tokenizer.name}")  # cl100k_base

# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer.encode(text))

토크나이저: cl100k_base


In [7]:
# 청크 샘플 검증
print(f"총 청크 수: {len(chunks)}")
print(f"평균 청크 길이: {sum(len(c.page_content) for c in chunks) // len(chunks)}자")
print()
for i, chunk in enumerate(chunks[:3]):
    print(f"[청크 {i+1}] {len(chunk.page_content)}자")
    print(chunk.page_content[:200])
    print("---")

총 청크 수: 140
평균 청크 길이: 462자

[청크 1] 307자
```
국채전문유통시장(KTS)
접속인프라 및 관리체계 개선 안내자료

2023. 6

한국거래소
채권시장부

회원 및 시스템 개발사 등의 인프라 개발에 필요한 시장 구조 등 전반적인 일반사항과 제도개선 사항을 포함하고 있습니다.
세부 제도 및 개발 방향은 변경될 수 있습니다.
```

---

### 2 주요 추진 일정

- 회원사의 시스템 개발 부담 및
---
[청크 2] 562자
#### 회원사 시스템 개발 일정(안)

| 구분          | I/F 일반매매 | I/F 시장조정 | 안전장치 |
| -------------- | ------------- | ------------- | --------- |
| 1. 사전개발 참여(~'23년말) | ●           | -           | -         |
| 2. 본
---
[청크 3] 589자
2. (본개발) I/F 일반 매매기능 등 + 시장조정 및 안전장치 개발
   - 본개발 사항의 가동시기는 '24년 상반기(1차 정기개편) 예정

3. (후발 개발) '24년 상반기 본개발 이후에 개발하는 유형
   - 구축조건 개발기한은 현재 미정이나 UI기능을 전지적으로 축소 예정(전체 회원사 개발일정 등 고려하여 추후 사전 공지)

- 개발 기간 중에는
---


`(2) 컨텍스트 생성`

각 청크에 대해 맥락 설명을 생성합니다.

In [8]:
import json
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# ── 컨텍스트 생성 LLM (저렴한 모델)
context_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# ── Anthropic 스타일 컨텍스트 생성 프롬프트
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 검색 전문가입니다.
주어진 청크가 전체 문서에서 어떤 위치에 있고 무엇에 관한 내용인지 한국어로 간결하게 설명하세요.
설명은 50-100자 이내로 작성하세요.
오직 맥락 설명만 출력하고 다른 내용은 추가하지 마세요."""),
    ("user", """<document>
{whole_document}
</document>

위 문서에서 아래 청크의 맥락을 설명해주세요:

<chunk>
{chunk_content}
</chunk>

맥락 설명:""")
])

context_chain = context_prompt | context_llm | StrOutputParser()

def generate_contexts(chunks, whole_document: str) -> list[str]:
    """각 청크의 맥락을 배치로 생성합니다."""
    inputs = [
        {"whole_document": whole_document, "chunk_content": chunk.page_content}
        for chunk in chunks
    ]
    return context_chain.batch(inputs, config={"max_concurrency": 5})

# ── 컨텍스트 캐시 (재실행 시 API 재호출 방지 + 평가 재현성 확보)
CONTEXTS_CACHE = Path("./contexts_cache.json")

if CONTEXTS_CACHE.exists():
    contexts = json.loads(CONTEXTS_CACHE.read_text(encoding="utf-8"))
    print(f"컨텍스트 캐시 로드: {len(contexts)}개 (재생성 건너뜀, API 비용 0)")
else:
    whole_document = docs[0].page_content
    print(f"컨텍스트 생성 시작: {len(chunks)}개 청크 (gpt-4.1-nano)")
    print(f"(전체 문서 62K자 × {len(chunks)}호출 ≈ 약 $0.3 이내 예상)")
    contexts = generate_contexts(chunks, whole_document)
    CONTEXTS_CACHE.write_text(
        json.dumps(contexts, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"컨텍스트 캐시 저장: {CONTEXTS_CACHE}")

print(f"\n컨텍스트 완료: {len(contexts)}개")

# ── contextual_chunks 생성 (맥락 + 원본 결합)
whole_document = docs[0].page_content
contextual_chunks = []
for i, (chunk, ctx) in enumerate(zip(chunks, contexts)):
    contextual_chunks.append(Document(
        page_content=f"[맥락] {ctx}\n\n{chunk.page_content}",
        metadata={
            **chunk.metadata,
            "chunk_id": i,
            "original_content": chunk.page_content,
            "context": ctx,
        }
    ))

print(f"Contextual 청크 생성 완료: {len(contextual_chunks)}개")
print(f"\n[샘플 - 청크 0]")
print(contextual_chunks[0].page_content[:300])

컨텍스트 생성 시작: 140개 청크 (gpt-4.1-nano)
(전체 문서 62K자 × 140호출 ≈ 약 $0.3 이내 예상)
컨텍스트 캐시 저장: contexts_cache.json

컨텍스트 완료: 140개
Contextual 청크 생성 완료: 140개

[샘플 - 청크 0]
[맥락] 국채시장(KTS)의 인프라 개선 안내와 2023년 6월 발표, 회원사 시스템 개발 단계별 추진 일정과 전환 계획을 설명하는 내용입니다.

```
국채전문유통시장(KTS)
접속인프라 및 관리체계 개선 안내자료

2023. 6

한국거래소
채권시장부

회원 및 시스템 개발사 등의 인프라 개발에 필요한 시장 구조 등 전반적인 일반사항과 제도개선 사항을 포함하고 있습니다.
세부 제도 및 개발 방향은 변경될 수 있습니다.
```

---

### 2 주요 추진 일정

- 회원사의 시스템 개발 부담 및 수용성을 고려해 단계적 개발 및


`(3) 하이브리드 검색 및 평가`

Embedding + BM25 + Reranker 파이프라인을 구성하고 성능을 평가합니다.

In [9]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from kiwipiepy import Kiwi

# ── 1. 벡터 저장소 생성 (기존 컬렉션 먼저 삭제 → 중복 append 방지)
for name in ["kts_normal", "kts_contextual"]:
    try:
        Chroma(collection_name=name, embedding_function=embeddings_model).delete_collection()
        print(f"  기존 컬렉션 삭제: {name}")
    except Exception:
        pass

print("벡터 저장소 생성 중...")
normal_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    collection_name="kts_normal",
)
contextual_vectorstore = Chroma.from_documents(
    documents=contextual_chunks,
    embedding=embeddings_model,
    collection_name="kts_contextual",
)
print(f"  일반:       {normal_vectorstore._collection.count()}개 청크")
print(f"  Contextual: {contextual_vectorstore._collection.count()}개 청크")

# ── 2. BM25 Retriever (Kiwi 한국어 형태소 분석기)
kiwi = Kiwi()
for word in ['국채전문유통시장', 'KTS', '시장조성', '조성호가', '조성기관',
             '드롭카피', '일중RP', '사전개발', '본개발', '접속인프라']:
    kiwi.add_user_word(word, 'NNP')

def kiwi_tokenize(text):
    return [t.form for t in kiwi.tokenize(text)]

normal_bm25 = BM25Retriever.from_documents(
    chunks, preprocess_func=kiwi_tokenize, k=3
)
contextual_bm25 = BM25Retriever.from_documents(
    contextual_chunks, preprocess_func=kiwi_tokenize, k=3
)
print("BM25 Retriever 생성 완료 (Kiwi)")

# ── 3. 하이브리드 Retriever (Embedding + BM25, RRF 앙상블)
normal_retriever     = normal_vectorstore.as_retriever(search_kwargs={"k": 3})
contextual_retriever = contextual_vectorstore.as_retriever(search_kwargs={"k": 3})

normal_hybrid = EnsembleRetriever(
    retrievers=[normal_retriever, normal_bm25], weights=[0.5, 0.5]
)
contextual_hybrid = EnsembleRetriever(
    retrievers=[contextual_retriever, contextual_bm25], weights=[0.5, 0.5]
)
print("하이브리드 Retriever 생성 완료")

# ── 4. 성능 평가 (HitRate@3, MRR@3)
#
# 평가 쿼리 설계 기준:
#   - [키워드형] query에 정답 키워드가 포함된 직접 질문  → BM25에 유리
#   - [맥락형]   query에 정답 키워드가 없고 문서 맥락을 알아야 연결되는 질문
#               → Contextual Retrieval이 우위를 보여야 하는 쿼리
#
eval_set = [
    # 키워드형 (5개) ─ BM25도 충분히 처리 가능
    {"query": "드롭카피를 통해 수신할 수 있는 정보 종류는?",       "expected": "드롭카피",  "type": "keyword"},
    {"query": "일중RP 거래 방식은?",                               "expected": "일중RP",    "type": "keyword"},
    {"query": "세션 변경 신청서 제출 방법은?",                      "expected": "세션",      "type": "keyword"},
    {"query": "사전개발 참여 회원사가 해야 하는 것은?",              "expected": "사전개발",  "type": "keyword"},
    {"query": "조성호가 제출 의무가 있는 기관은?",                  "expected": "조성호가",  "type": "keyword"},

    # 맥락형 (5개) ─ query에 정답 키워드 없음, 문서 전체 맥락 필요
    {"query": "KRX가 이 제도 개선을 추진하는 배경은?",              "expected": "시장조성",  "type": "contextual"},
    {"query": "회원사가 선택할 수 있는 개발 참여 방식은?",           "expected": "사전개발",  "type": "contextual"},
    {"query": "기존 단말 방식은 언제까지 계속 쓸 수 있나?",          "expected": "병행",      "type": "contextual"},
    {"query": "외국인 투자자 거래를 위해 무엇을 정비해야 하나?",      "expected": "외국인",    "type": "contextual"},
    {"query": "본격 가동 이전에 참여할 수 있는 개발 단계는?",        "expected": "사전개발",  "type": "contextual"},
]

def evaluate(retriever, eval_set: list, k: int = 3) -> dict:
    hit, mrr_total = 0, 0.0
    for item in eval_set:
        results = retriever.invoke(item["query"])[:k]
        all_text = " ".join([
            doc.metadata.get("original_content", doc.page_content) for doc in results
        ])
        if item["expected"] in all_text:
            hit += 1
            for rank, doc in enumerate(results, 1):
                doc_text = doc.metadata.get("original_content", doc.page_content)
                if item["expected"] in doc_text:
                    mrr_total += 1 / rank
                    break
    n = len(eval_set)
    return {"hit_rate": hit / n, "mrr": mrr_total / n}

def evaluate_by_type(retriever, eval_set, k=3):
    keyword_set = [q for q in eval_set if q["type"] == "keyword"]
    ctx_set     = [q for q in eval_set if q["type"] == "contextual"]
    return {
        "overall":    evaluate(retriever, eval_set, k),
        "keyword":    evaluate(retriever, keyword_set, k),
        "contextual": evaluate(retriever, ctx_set, k),
    }

print("\n" + "="*65)
print("성능 평가 (HitRate@3, MRR@3)  |  키워드형 5개 + 맥락형 5개")
print("="*65)

retrievers = {
    "일반 Embedding":        normal_retriever,
    "일반 BM25":             normal_bm25,
    "일반 Hybrid":           normal_hybrid,
    "Contextual Embedding":  contextual_retriever,
    "Contextual BM25":       contextual_bm25,
    "Contextual Hybrid":     contextual_hybrid,
}

all_results = {}
for name, retriever in retrievers.items():
    all_results[name] = evaluate_by_type(retriever, eval_set)
    r = all_results[name]["overall"]
    print(f"  {name:25s}: HitRate={r['hit_rate']:.1%}  MRR={r['mrr']:.3f}")

# ── 5. 요약: HitRate + MRR 비교
print("\n" + "="*65)
print("일반 vs Contextual 비교 요약")
print("="*65)

normal_keys = ["일반 Embedding", "일반 BM25", "일반 Hybrid"]
ctx_keys    = ["Contextual Embedding", "Contextual BM25", "Contextual Hybrid"]

normal_avg_hr  = sum(all_results[k]["overall"]["hit_rate"] for k in normal_keys) / 3
ctx_avg_hr     = sum(all_results[k]["overall"]["hit_rate"] for k in ctx_keys)    / 3
normal_avg_mrr = sum(all_results[k]["overall"]["mrr"]      for k in normal_keys) / 3
ctx_avg_mrr    = sum(all_results[k]["overall"]["mrr"]      for k in ctx_keys)    / 3

print(f"{'':30s}  {'HitRate':>8}  {'MRR':>8}")
print(f"{'일반 검색 평균':30s}  {normal_avg_hr:>8.1%}  {normal_avg_mrr:>8.3f}")
print(f"{'Contextual 검색 평균':30s}  {ctx_avg_hr:>8.1%}  {ctx_avg_mrr:>8.3f}")

hr_diff  = ctx_avg_hr  - normal_avg_hr
mrr_diff = ctx_avg_mrr - normal_avg_mrr
print(f"\n→ HitRate  변화: {hr_diff:+.1%}")
print(f"→ MRR      변화: {mrr_diff:+.3f}  ({mrr_diff/normal_avg_mrr*100:+.1f}%)")

# ── 6. 맥락형 쿼리만 별도 비교 (Contextual의 진가)
print("\n" + "="*65)
print("맥락형 쿼리만 비교 (Contextual Retrieval 효과가 두드러지는 구간)")
print("="*65)
for name in retrievers:
    r = all_results[name]["contextual"]
    tag = "← Contextual" if "Contextual" in name else ""
    print(f"  {name:25s}: HitRate={r['hit_rate']:.1%}  MRR={r['mrr']:.3f}  {tag}")

  기존 컬렉션 삭제: kts_normal
  기존 컬렉션 삭제: kts_contextual
벡터 저장소 생성 중...
  일반:       140개 청크
  Contextual: 140개 청크
BM25 Retriever 생성 완료 (Kiwi)
하이브리드 Retriever 생성 완료

성능 평가 (HitRate@3, MRR@3)  |  키워드형 5개 + 맥락형 5개
  일반 Embedding             : HitRate=60.0%  MRR=0.600
  일반 BM25                  : HitRate=70.0%  MRR=0.583
  일반 Hybrid                : HitRate=60.0%  MRR=0.600
  Contextual Embedding     : HitRate=70.0%  MRR=0.633
  Contextual BM25          : HitRate=60.0%  MRR=0.550
  Contextual Hybrid        : HitRate=60.0%  MRR=0.600

일반 vs Contextual 비교 요약
                                 HitRate       MRR
일반 검색 평균                           63.3%     0.594
Contextual 검색 평균                   63.3%     0.594

→ HitRate  변화: +0.0%
→ MRR      변화: +0.000  (+0.0%)

맥락형 쿼리만 비교 (Contextual Retrieval 효과가 두드러지는 구간)
  일반 Embedding             : HitRate=60.0%  MRR=0.600  
  일반 BM25                  : HitRate=60.0%  MRR=0.500  
  일반 Hybrid                : HitRate=60.0%  MRR=0.600  
  Contextual Embedding  

---

## Gradio UI — KTS/KRX 문서 RAG 챗봇

**W3 요소** (메모리 관리) + **W4 요소** (Contextual Retrieval, Hybrid Search, Reranker) 통합

| 설정 항목 | 출처 | 설명 |
|---|---|---|
| 메모리 방식 (RunnableWithMessageHistory / create_agent) | W3 수업 2·3.1 | SQLite 영구 저장 vs InMemorySaver |
| 대화 이력 자동 요약 | W3 수업 2.4 | 임계값 초과 시 LLM 요약 |
| 답변 품질 평가 | W3 수업 | AnswerQualityOutput (structured output) |
| 검색 방식 선택 | W4 | 일반/Contextual × Embedding/BM25/Hybrid |
| BM25 가중치 슬라이더 | W4 중급 | 가중치 조합 실험 (0.3:0.7, 0.5:0.5 등) |
| Reranker | W4 고급 | CrossEncoder 재순위화 (⚠️ CPU 느림) |

In [10]:
import os
import gradio as gr
import uuid
import sqlite3
from pathlib import Path
from typing import Generator, List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, SystemMessage, AIMessageChunk
)
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field


# ══════════════════════════════════════════════════════════════
# 임베딩 모델 + 벡터 저장소 (디스크 캐시 포함)
# ══════════════════════════════════════════════════════════════

MODEL_KEY_MAP = {
    "OpenAI text-embedding-3-small": "openai",
    "HuggingFace BAAI/bge-m3":       "bge_m3",
}

KTS_DB_ROOT = Path("./kts_vector_db")


def build_embedding_model(name: str):
    if name == "HuggingFace BAAI/bge-m3":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
    return OpenAIEmbeddings(model="text-embedding-3-small")


_vs_memory_cache: dict = {}


def get_or_build_vectorstore(model_name: str, vs_type: str,
                              docs, collection_key: str):
    cache_key = (model_name, vs_type, collection_key)
    if cache_key in _vs_memory_cache:
        print(f"  [캐시] {vs_type} {MODEL_KEY_MAP.get(model_name, '?')}/{collection_key}")
        return _vs_memory_cache[cache_key]

    model_key = MODEL_KEY_MAP.get(model_name, "openai")
    emb = build_embedding_model(model_name)

    if vs_type == "Chroma":
        persist_dir = str(KTS_DB_ROOT / "chroma" / model_key / collection_key)
        exists = (Path(persist_dir) / "chroma.sqlite3").exists()
        if exists:
            print(f"  [로드] Chroma {model_key}/{collection_key} (디스크)")
            vs = Chroma(
                persist_directory=persist_dir,
                embedding_function=emb,
                collection_name=f"kts_{collection_key}_{model_key}",
            )
        else:
            print(f"  [임베딩] Chroma {model_key}/{collection_key} (최초 — 디스크 저장)")
            vs = Chroma.from_documents(
                documents=docs,
                embedding=emb,
                collection_name=f"kts_{collection_key}_{model_key}",
                persist_directory=persist_dir,
            )
    else:  # FAISS
        from langchain_community.vectorstores import FAISS
        faiss_dir = str(KTS_DB_ROOT / "faiss" / model_key / collection_key)
        exists = (Path(faiss_dir) / "index.faiss").exists()
        if exists:
            print(f"  [로드] FAISS {model_key}/{collection_key} (디스크)")
            vs = FAISS.load_local(faiss_dir, emb, allow_dangerous_deserialization=True)
        else:
            print(f"  [임베딩] FAISS {model_key}/{collection_key} (최초 — 디스크 저장)")
            vs = FAISS.from_documents(docs, emb)
            Path(faiss_dir).mkdir(parents=True, exist_ok=True)
            vs.save_local(faiss_dir)

    _vs_memory_cache[cache_key] = vs
    return vs


# ══════════════════════════════════════════════════════════════
# W3 수업 2.2·2.4 — SummarizedSQLiteHistory
# ══════════════════════════════════════════════════════════════

class SummarizedSQLiteHistory(BaseChatMessageHistory):
    def __init__(self, session_id: str, db_path: str = "kts_chat_history.db",
                 summary_threshold: int = 6, llm=None):
        self.session_id = session_id
        self.db_path = db_path
        self.summary_threshold = summary_threshold
        self.llm = llm or ChatOpenAI(model="gpt-4.1-nano", temperature=0)
        self._create_table()

    def _create_table(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS messages (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    msg_type   TEXT NOT NULL,
                    content    TEXT NOT NULL,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)

    @property
    def messages(self) -> List[BaseMessage]:
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT msg_type, content FROM messages WHERE session_id=? ORDER BY created_at",
                (self.session_id,)
            ).fetchall()
        type_map = {"HumanMessage": HumanMessage, "AIMessage": AIMessage}
        return [type_map.get(t, SystemMessage)(content=c) for t, c in rows]

    def add_messages(self, new_msgs: List[BaseMessage]) -> None:
        with sqlite3.connect(self.db_path) as conn:
            conn.executemany(
                "INSERT INTO messages (session_id, msg_type, content) VALUES (?,?,?)",
                [(self.session_id, m.__class__.__name__, m.content) for m in new_msgs]
            )
        if len(self.messages) >= self.summary_threshold:
            self._summarize_and_compress(self.messages)

    def _summarize_and_compress(self, msgs: List[BaseMessage]):
        to_summarize, to_keep = msgs[:-2], msgs[-2:]
        if not to_summarize:
            return
        summary = self.llm.invoke([
            SystemMessage(content="다음 대화를 핵심 내용 위주로 한국어로 간결하게 요약하세요."),
            *to_summarize,
        ])
        compressed = [SystemMessage(content=f"[이전 대화 요약]\n{summary.content}"), *to_keep]
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM messages WHERE session_id=?", (self.session_id,))
            conn.executemany(
                "INSERT INTO messages (session_id, msg_type, content) VALUES (?,?,?)",
                [(self.session_id, m.__class__.__name__, m.content) for m in compressed]
            )

    def clear(self) -> None:
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM messages WHERE session_id=?", (self.session_id,))


_history_cache: dict = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in _history_cache:
        _history_cache[session_id] = SummarizedSQLiteHistory(session_id=session_id)
    return _history_cache[session_id]


# ══════════════════════════════════════════════════════════════
# W3 수업 — 답변 품질 평가
# ══════════════════════════════════════════════════════════════

class AnswerQualityOutput(BaseModel):
    is_sufficient: bool = Field(description="답변이 충분하면 True, 불충분이면 False")
    reason: str         = Field(description="평가 이유 한 문장")


# ══════════════════════════════════════════════════════════════
# W3+W4 — KTSRAGSystem
# ══════════════════════════════════════════════════════════════

class KTSRAGSystem:

    def __init__(self, llm, eval_llm, retriever, top_k: int = 3):
        self.llm       = llm
        self.eval_llm  = eval_llm
        self.retriever = retriever
        self.top_k     = top_k
        self._build_chain()

    def _build_chain(self):
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 한국거래소(KRX) 국채전문유통시장(KTS) 접속인프라 및 관리체계 개선 전문가입니다.\n"
             "주어진 문서는 OCR 추출 텍스트라 표·구분기호 포맷이 불규칙할 수 있습니다. "
             "포맷에 관계없이 내용을 파악하여 관련 정보가 조금이라도 있으면 적극적으로 활용해 답변하세요.\n"
             "문서를 충분히 확인했음에도 완전히 관련 내용이 없을 때만 "
             "'문서에서 해당 정보를 찾을 수 없습니다.'라고 답변하세요.\n\n"
             "[참고 문서]\n{context}"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ])
        self.chain_with_history = RunnableWithMessageHistory(
            prompt | self.llm | StrOutputParser(),
            get_session_history,
            input_messages_key="question",
            history_messages_key="history",
        )

    def update_retriever(self, retriever, top_k: int = None):
        self.retriever = retriever
        if top_k is not None:
            self.top_k = top_k

    def _format_docs(self, docs: List) -> str:
        parts = []
        for doc in docs:
            content = doc.metadata.get("original_content", doc.page_content)
            ctx = doc.metadata.get("context", "")
            parts.append(f"[맥락] {ctx}\n{content}" if ctx else content)
        return "\n\n---\n\n".join(parts)

    def _format_source_line(self, doc) -> str:
        """참조 문서 한 줄 요약 — context 메타 우선, 없으면 원본 첫 줄"""
        ctx = doc.metadata.get("context", "")
        if ctx:
            return ctx[:80]
        content = doc.metadata.get("original_content", doc.page_content)
        first_line = content.strip().split("\n")[0]
        return first_line[:80]

    def _evaluate_quality(self, question: str, answer: str) -> AnswerQualityOutput:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "답변이 질문에 충분히 답하는지 평가하세요."),
            ("human", "질문: {question}\n\n답변: {answer}")
        ])
        return (prompt | self.eval_llm.with_structured_output(AnswerQualityOutput)).invoke(
            {"question": question, "answer": answer}
        )

    def generate_answer(self, message: str, session_id: str) -> Generator[str, None, None]:
        all_docs = self.retriever.invoke(message)
        docs = all_docs[:self.top_k]   # Issue 4: Hybrid가 2×k 반환하는 것을 top_k로 제한
        if not docs:
            yield "관련 문서를 찾을 수 없습니다."
            return

        context = self._format_docs(docs)
        config  = {"configurable": {"session_id": session_id}}
        full_answer = ""
        try:
            for chunk in self.chain_with_history.stream(
                {"context": context, "question": message}, config=config
            ):
                full_answer += chunk
                yield full_answer

            quality = self._evaluate_quality(message, full_answer)

            # Issue 5: context 메타 기반 참조 문서 요약
            sources = "\n".join([
                f"- 📄 {self._format_source_line(doc)}" for doc in docs
            ])
            suffix = f"\n\n---\n📚 참조 문서 {len(docs)}개\n{sources}"

            # Issue 3: 답변 교체 없이 원본 유지 + 신뢰도 낮으면 경고 한 줄만 추가
            if not quality.is_sufficient:
                suffix += f"\n\n⚠️ 낮은 신뢰도: {quality.reason}"
            yield f"{full_answer}{suffix}"

        except Exception as e:
            yield f"오류 발생: {e}"


# ══════════════════════════════════════════════════════════════
# W4 — 동적 Retriever 빌더
# ══════════════════════════════════════════════════════════════

_reranker_model = None

def get_reranker():
    global _reranker_model
    if _reranker_model is None:
        print("Reranker 모델 로딩 중 (최초 1회)...")
        ce = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
        _reranker_model = CrossEncoderReranker(model=ce, top_n=3)
        print("Reranker 로딩 완료")
    return _reranker_model


def build_ui_retriever(model_name: str, vs_type: str,
                        retriever_type: str, bm25_weight: float,
                        top_k: int, use_reranker: bool):
    emb_weight = round(1.0 - bm25_weight, 2)
    k = int(top_k)

    normal_vs     = get_or_build_vectorstore(model_name, vs_type, chunks,            "normal")
    contextual_vs = get_or_build_vectorstore(model_name, vs_type, contextual_chunks, "contextual")

    nr = normal_vs.as_retriever(search_kwargs={"k": k})
    cr = contextual_vs.as_retriever(search_kwargs={"k": k})
    nb = BM25Retriever.from_documents(chunks,            preprocess_func=kiwi_tokenize, k=k)
    cb = BM25Retriever.from_documents(contextual_chunks, preprocess_func=kiwi_tokenize, k=k)

    retriever_map = {
        "일반 Embedding":        nr,
        "일반 BM25":             nb,
        "일반 Hybrid":           EnsembleRetriever(retrievers=[nr, nb], weights=[emb_weight, bm25_weight]),
        "Contextual Embedding":  cr,
        "Contextual BM25":       cb,
        "Contextual Hybrid":     EnsembleRetriever(retrievers=[cr, cb], weights=[emb_weight, bm25_weight]),
    }
    base = retriever_map[retriever_type]

    if use_reranker:
        return ContextualCompressionRetriever(
            base_compressor=get_reranker(),
            base_retriever=base,
        )
    return base


# ══════════════════════════════════════════════════════════════
# W3 수업 3.1 — create_agent + InMemorySaver
# ══════════════════════════════════════════════════════════════

_ui_llm      = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
_ui_eval_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

_rag_system          = KTSRAGSystem(llm=_ui_llm, eval_llm=_ui_eval_llm,
                                     retriever=contextual_hybrid, top_k=3)
_agent_checkpointer  = InMemorySaver()
_agent_retriever_ref = {"retriever": contextual_hybrid}


@tool
def search_kts_document(query: str) -> str:
    """KTS 문서에서 관련 정보를 검색합니다."""
    docs = _agent_retriever_ref["retriever"].invoke(query)
    if not docs:
        return "관련 문서를 찾을 수 없습니다."
    return "\n\n---\n\n".join([
        doc.metadata.get("original_content", doc.page_content) for doc in docs
    ])


_agent = create_agent(
    model="gpt-4.1-nano",
    tools=[search_kts_document],
    system_prompt=(
        "당신은 한국거래소(KRX) 국채전문유통시장(KTS) 전문 어시스턴트입니다. "
        "반드시 search_kts_document 도구로 문서를 검색한 후 답변하세요. "
        "문서에 없는 내용은 '문서에 근거 없음'이라고 답변하세요."
    ),
    checkpointer=_agent_checkpointer,
)


def generate_with_agent(message: str, thread_id: str) -> Generator[str, None, None]:
    config      = {"configurable": {"thread_id": thread_id}}
    full_answer = ""
    try:
        for chunk, _ in _agent.stream(
            {"messages": [{"role": "user", "content": message}]},
            config=config,
            stream_mode="messages",
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                full_answer += chunk.content
                yield full_answer
        if not full_answer:
            yield "죄송합니다. 답변을 생성하지 못했습니다."
    except Exception as e:
        yield f"오류 발생: {e}"


print("✅ KTSRAGSystem + create_agent 초기화 완료")
print(f"   벡터 DB 루트: {KTS_DB_ROOT.resolve()}")

✅ KTSRAGSystem + create_agent 초기화 완료
   벡터 DB 루트: /Users/greenpianorabbit/edu/modu_llm5/002_etfbot/homework/kts_vector_db


In [11]:
_last_ui_settings = None

green_theme = gr.themes.Soft(
    primary_hue="green",
    secondary_hue="emerald",
    neutral_hue="slate",
)

with gr.Blocks(theme=green_theme, title="KTS/KRX RAG 챗봇") as demo:

    gr.Markdown(
        "# 📊 KTS/KRX 문서 RAG 챗봇\n"
        "국채전문유통시장(KTS) 접속인프라 및 관리체계 개선 안내자료 | "
        "W3 메모리 관리 + W4 Contextual Retrieval"
    )

    session_id_state = gr.State(value=str(uuid.uuid4()))

    with gr.Row():

        # ── 왼쪽: 설정 패널 ──────────────────────────────────────
        with gr.Column(scale=1, min_width=300):

            gr.Markdown("### 🧠 메모리 방식 (W3)")
            memory_mode = gr.Radio(
                choices=[
                    "RunnableWithMessageHistory (레거시/고기능)",
                    "create_agent (권장)",
                ],
                value="RunnableWithMessageHistory (레거시/고기능)",
                label="",
                info="레거시: SQLite 영구 저장 + 자동 요약 + 품질 평가 | 권장: InMemorySaver",
            )

            gr.Markdown("---\n### 🤖 임베딩 & 저장소")
            embedding_model = gr.Dropdown(
                choices=["OpenAI text-embedding-3-small", "HuggingFace BAAI/bge-m3"],
                value="OpenAI text-embedding-3-small",
                label="임베딩 모델",
                info="첫 선택 시 임베딩 후 디스크 저장 → 이후 로드만",
            )
            vs_type = gr.Dropdown(
                choices=["Chroma", "FAISS"],
                value="Chroma",
                label="벡터 저장소 타입",
            )

            gr.Markdown("---\n### 🔍 검색 설정 (W4)")
            retriever_type = gr.Dropdown(
                choices=[
                    "일반 Embedding", "일반 BM25", "일반 Hybrid",
                    "Contextual Embedding", "Contextual BM25", "Contextual Hybrid",
                ],
                value="Contextual Hybrid",
                label="검색 방식",
            )
            bm25_weight = gr.Slider(
                0.0, 1.0, value=0.5, step=0.1,
                label="⚖️ BM25 가중치 (Embedding = 1 - BM25)",
                info="Hybrid 선택 시 유효 | 0.3/0.5/0.7 실험",
            )
            top_k = gr.Slider(1, 10, value=3, step=1, label="📦 Top-K")
            use_reranker = gr.Checkbox(
                value=False,
                label="🔄 Reranker (CrossEncoder)",
                info="⚠️ Intel Mac CPU 느림 — 최초 실행 시 모델 다운로드",
            )

        # ── 오른쪽: 채팅 ─────────────────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                type="tuples",
                height=520,
                show_copy_button=True,
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="KTS/KRX 문서에 대해 질문하세요...",
                    show_label=False,
                    scale=5,
                    lines=1,
                )
                submit_btn = gr.Button("전송", variant="primary", scale=1, min_width=80)

            with gr.Row():
                clear_btn = gr.Button("🗑️ 대화 초기화", size="sm", variant="secondary")

            gr.Examples(
                examples=[
                    ["회원사 시스템 개발 단계는 어떻게 되나요?"],
                    ["시장조성기관의 역할과 의무는 무엇인가요?"],
                    ["드롭카피를 통해 수신할 수 있는 정보는?"],
                    ["KRX가 이 제도 개선을 추진하는 배경은?"],
                    ["기존 단말 방식은 언제까지 사용 가능한가요?"],
                ],
                inputs=msg_input,
                label="예시 질문",
            )

    # ── 이벤트 핸들러 ─────────────────────────────────────────

    def respond(message, history, session_id,
                memory_mode, embedding_model, vs_type,
                retriever_type, bm25_weight, top_k, use_reranker):
        global _last_ui_settings
        if not message or not message.strip():
            yield history, ""
            return

        cur = {
            "emb": embedding_model, "vs": vs_type, "type": retriever_type,
            "bm25": bm25_weight, "k": top_k, "rerank": use_reranker,
        }
        if cur != _last_ui_settings:
            print(f"\n[설정 변경] {cur}")
            new_retriever = build_ui_retriever(
                embedding_model, vs_type, retriever_type,
                bm25_weight, int(top_k), use_reranker
            )
            _rag_system.update_retriever(new_retriever, top_k=int(top_k))
            _agent_retriever_ref["retriever"] = new_retriever
            _last_ui_settings = cur

        # tuples 포맷: [[user_msg, bot_msg], ...]
        history = history + [[message, None]]
        yield history, ""

        if "create_agent" in memory_mode:
            gen = generate_with_agent(message, session_id)
        else:
            gen = _rag_system.generate_answer(message, session_id)

        for chunk in gen:
            history[-1][1] = chunk
            yield history, ""

    inputs = [
        msg_input, chatbot, session_id_state,
        memory_mode, embedding_model, vs_type,
        retriever_type, bm25_weight, top_k, use_reranker,
    ]
    outputs = [chatbot, msg_input]

    submit_btn.click(respond, inputs, outputs)
    msg_input.submit(respond, inputs, outputs)
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_input])

demo.launch(show_api=False)


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [ ]:
# demo.close()


[설정 변경] {'emb': 'OpenAI text-embedding-3-small', 'vs': 'Chroma', 'type': 'Contextual Hybrid', 'bm25': 0.5, 'k': 3, 'rerank': False}
  [로드] Chroma openai/normal (디스크)
  [로드] Chroma openai/contextual (디스크)
